In [ ]:
!pip install pyspark -q
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, datetime, gc
from pyspark.sql import SparkSession
from pyspark.ml.recommendation import ALS
from pyspark.ml.feature import StringIndexer
from pyspark.sql import functions as F
from pyspark.mllib.evaluation import RankingMetrics

# 1. Cấu hình đường dẫn
BASE_PATH = "/content/drive/MyDrive/HM-DATA/"
INPUT_FILE = BASE_PATH + "processed/cleaned_transactions.parquet"
OUTPUT_DIR = BASE_PATH + "outputs/"
CHECKPOINT_DIR = BASE_PATH + "spark_checkpoints/"

os.makedirs(OUTPUT_DIR + "candidates/", exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# 2. Khởi tạo Spark
spark = SparkSession.builder \
    .appName("HM_ALS_Final_W8") \
    .config("spark.driver.memory", "10g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setCheckpointDir(CHECKPOINT_DIR)
print("✅ Spark Ready! Hệ thống sẵn sàng tạo ứng viên Tuần 8.")

Mounted at /content/drive
✅ Spark Ready! Hệ thống sẵn sàng tạo ứng viên Tuần 8.


In [ ]:
# 1. Đọc dữ liệu
df = spark.read.parquet(INPUT_FILE)
max_date = df.select(F.max("t_dat_date")).collect()[0][0]

# 2. Định nghĩa mốc thời gian mới
# Tuần 8 bắt đầu từ 7 ngày cuối cùng của dữ liệu
test_start_date = max_date - datetime.timedelta(days=7)

# Lấy 7 tuần lịch sử để train (bao gồm cả tuần Validation cũ)
train_retrieval = df.filter(F.col("t_dat_date") < F.lit(test_start_date))
# Tuần 8 dùng để đánh giá Recall/MAP thật
test_ranking = df.filter(F.col("t_dat_date") >= F.lit(test_start_date))

# 3. String Indexing (Fit trên toàn bộ 7 tuần để không bỏ sót User/Item nào)
u_model = StringIndexer(inputCol="customer_id", outputCol="user_idx").setHandleInvalid("skip").fit(train_retrieval)
i_model = StringIndexer(inputCol="article_id", outputCol="item_idx").setHandleInvalid("skip").fit(train_retrieval)

# Chuẩn bị đáp án Tuần 8 để đánh giá nội bộ
test_gt_indexed = i_model.transform(u_model.transform(test_ranking)) \
    .groupBy("user_idx") \
    .agg(F.collect_list("item_idx").alias("actual_item_idxs"))

print(f"📅 Giai đoạn Train (W1-7): Tính đến ngày {test_start_date}")
print(f"🎯 Giai đoạn Dự đoán (W8): Từ {test_start_date} đến nay")

📅 Giai đoạn Train (W1-7): Tính đến ngày 2020-09-15
🎯 Giai đoạn Dự đoán (W8): Từ 2020-09-15 đến nay


In [ ]:
BEST_DECAY = 0.1
# BEST_RANK = 80
BEST_RANK = 40
BEST_ALPHA = 40

print("🚀 Đang huấn luyện ALS trên dữ liệu 7 tuần...")

# Tính Ratings: Món đồ mua gần ngày bắt đầu Tuần 8 sẽ có điểm cao hơn
ratings = train_retrieval.withColumn("days_diff", F.datediff(F.lit(test_start_date), F.col("t_dat_date"))) \
                         .withColumn("weight", F.exp(-BEST_DECAY * F.col("days_diff"))) \
                         .groupBy("customer_id", "article_id") \
                         .agg(F.sum("weight").alias("rating"))

train_indexed = i_model.transform(u_model.transform(ratings)).checkpoint()

# Huấn luyện mô hình ALS
als_model = ALS(maxIter=15, rank=BEST_RANK, regParam=0.1, alpha=BEST_ALPHA,
                userCol="user_idx", itemCol="item_idx", ratingCol="rating",
                implicitPrefs=True, coldStartStrategy="drop", nonnegative=True).fit(train_indexed)

print("✅ Huấn luyện ALS Tuần 8 hoàn tất!")

🚀 Đang huấn luyện ALS trên dữ liệu 7 tuần...
✅ Huấn luyện ALS Tuần 8 hoàn tất!


In [ ]:
print("🎯 Đang tạo 100 ứng viên cho Tuần 8 và đánh giá...")

# Tạo 100 ứng viên cho những khách hàng có xuất hiện trong Tuần 8
val_recs = als_model.recommendForUserSubset(test_gt_indexed.select("user_idx"), 100).cache()

# Tính MAP@12 thực tế trên Tuần 8
eval_rdd = val_recs.join(test_gt_indexed, "user_idx") \
    .select(F.col("recommendations.item_idx").alias("p"), "actual_item_idxs") \
    .rdd.map(lambda r: (list(r[0]), list(r[1])))

map12 = RankingMetrics(eval_rdd.map(lambda x: (x[0][:12], x[1]))).meanAveragePrecision
print(f"🏆 MAP@12 THỰC TẾ TRÊN TUẦN 8: {map12:.6f}")

🎯 Đang tạo 100 ứng viên cho Tuần 8 và đánh giá...


/usr/local/lib/python3.12/dist-packages/pyspark/sql/context.py:157: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


🏆 MAP@12 THỰC TẾ TRÊN TUẦN 8: 0.012911


In [ ]:
print("💾 Đang giải mã Index sang ID gốc bằng Spark-Native (Tránh crash JVM)...")

# 1. Tạo bảng tra cứu trực tiếp từ Indexer (Không lôi list về Python)
# Chúng ta dùng chính model để transform lại tập ID duy nhất
user_map = u_model.transform(train_retrieval.select("customer_id").distinct()).select("customer_id", "user_idx")
item_map = i_model.transform(train_retrieval.select("article_id").distinct()).select("article_id", "item_idx")

# 2. Giải mã và chuẩn hóa mã 10 số cho Article_ID
# Sử dụng join trực tiếp trong Spark để giữ hiệu năng cao
val_recs_decoded = val_recs.select("user_idx", F.explode("recommendations").alias("rec")) \
    .select("user_idx", "rec.item_idx", F.col("rec.rating").alias("als_score")) \
    .join(user_map, "user_idx") \
    .join(item_map, "item_idx") \
    .withColumn("article_id", F.lpad(F.col("article_id").cast("string"), 10, "0")) \
    .select("customer_id", "article_id", "als_score")

# 3. Lưu file ứng viên Tuần 8 (Dự đoán tương lai)
OUTPUT_FILE_W8 = OUTPUT_DIR + "candidates/als_top100_W8_decoded.parquet"
val_recs_decoded.write.mode("overwrite").parquet(OUTPUT_FILE_W8)

# 4. Lưu lại Model và Indexer để dùng cho các bước sau
als_model.write().overwrite().save(OUTPUT_DIR + "models/als_model_W8")
u_model.write().overwrite().save(OUTPUT_DIR + "models/user_indexer_W8")
i_model.write().overwrite().save(OUTPUT_DIR + "models/item_indexer_W8")

print(f"🏆 CHÚC MỪNG LEADER! File ứng viên Tuần 8 đã được lưu an toàn tại: {OUTPUT_FILE_W8}")

💾 Đang giải mã Index sang ID gốc bằng Spark-Native (Tránh crash JVM)...
🏆 CHÚC MỪNG LEADER! File ứng viên Tuần 8 đã được lưu an toàn tại: /content/drive/MyDrive/HM-DATA/outputs/candidates/als_top100_W8_decoded.parquet
